#### 本教程演示了如何使用Adversarial Debiasing算法来学习公平的分类器。

对抗性去偏见，Adversarial debiasing[1]是一种处理中技术，此方法通过学习分类器来最大限度地提高预测准确性，同时降低对手从预测中确定受保护属性的能力。这种方法可以得到一个公平的分类器，因为预测不可能携带任何可被对手利用的群体歧视信息。在这个教程中，你将了解如何使用这种算法来学习有公平性约束和无公平性约束的模型，并将它们应用于 Adult 数据集。

In [1]:
%matplotlib inline

# 加载需要的库
import sys
sys.path.append("../")
from aif360.datasets import BinaryLabelDataset
from aif360.datasets import AdultDataset, GermanDataset, CompasDataset
from aif360.datasets import OULADataset

from aif360.datasets import StuperDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.metrics import ClassificationMetric
from aif360.metrics.utils import compute_boolean_conditioning_vector

from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import load_preproc_data_oula
from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import load_preproc_data_stuper
from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import load_preproc_data_duolingo

from aif360.algorithms.inprocessing.adversarial_debiasing import AdversarialDebiasing

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.metrics import accuracy_score

from IPython.display import Markdown, display
import matplotlib.pyplot as plt

import tensorflow.compat.v1 as tf
tf.disable_eager_execution()

pip install 'aif360[LawSchoolGPA]'
C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\inFairness\utils\ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\inFairness\utils\ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted 

#### 读取数据集和设置选项

In [2]:
# 获取数据集，进行训练集和测试集的划分
dataset_orig = load_preproc_data_oula()

privileged_groups = [{'gender': 0}]
unprivileged_groups = [{'gender': 1}]

dataset_orig_train, dataset_orig_test = dataset_orig.split([0.7], shuffle=True)

In [3]:
# 打印出数据集的一些特征
display(Markdown("#### Training Dataset shape"))
print(dataset_orig_train.features.shape)
display(Markdown("#### Favorable and unfavorable labels"))
print(dataset_orig_train.favorable_label, dataset_orig_train.unfavorable_label)
display(Markdown("#### Protected attribute names"))
print(dataset_orig_train.protected_attribute_names)
display(Markdown("#### Privileged and unprivileged protected attribute values"))
print(dataset_orig_train.privileged_protected_attributes, 
      dataset_orig_train.unprivileged_protected_attributes)
display(Markdown("#### Dataset feature names"))
print(dataset_orig_train.feature_names)

#### Training Dataset shape

(114375, 38)


#### Favorable and unfavorable labels

1.0 0.0


#### Protected attribute names

['gender']


#### Privileged and unprivileged protected attribute values

[array([1.])] [array([0.])]


#### Dataset feature names

['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight', 'module_presentation_length', 'id_student', 'date_submitted', 'is_banked', 'score', 'gender', 'imd_band', 'num_of_prev_attempts', 'studied_credits', 'region=0', 'region=1', 'region=2', 'region=3', 'region=4', 'region=5', 'region=6', 'region=7', 'region=8', 'region=9', 'region=10', 'region=11', 'region=12', 'highest_education=0', 'highest_education=1', 'highest_education=2', 'highest_education=3', 'highest_education=4', 'age_band=0', 'age_band=1', 'age_band=2', 'disability=0', 'disability=1']


#### 原始训练数据的指标

In [4]:
# 原始数据集的指标
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_orig_train.mean_difference())
metric_orig_test = BinaryLabelDatasetMetric(dataset_orig_test, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_orig_test.mean_difference())

#### Original training dataset

Train set: Difference in mean outcomes between unprivileged and privileged groups = -0.004141
Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.005996


In [5]:
min_max_scaler = MaxAbsScaler()
dataset_orig_train.features = min_max_scaler.fit_transform(dataset_orig_train.features)
dataset_orig_test.features = min_max_scaler.transform(dataset_orig_test.features)
metric_scaled_train = BinaryLabelDatasetMetric(dataset_orig_train, 
                             unprivileged_groups=unprivileged_groups,
                             privileged_groups=privileged_groups)
# 缩放数据集 - 验证缩放是否不会影响组标签统计数据
display(Markdown("#### Scaled dataset - Verify that the scaling does not affect the group label statistics"))
print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_scaled_train.mean_difference())
metric_scaled_test = BinaryLabelDatasetMetric(dataset_orig_test, 
                             unprivileged_groups=unprivileged_groups,
                             privileged_groups=privileged_groups)
print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_scaled_test.mean_difference())


#### Scaled dataset - Verify that the scaling does not affect the group label statistics

Train set: Difference in mean outcomes between unprivileged and privileged groups = -0.004141
Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.005996


### 应用基于Adversarial Debiasing的处理中算法

In [6]:
# Learn parameters with debias set to True
# 在去偏见设置为True时学习参数
sess = tf.Session()
debiased_model = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='debiased_classifier',
                          debias=True,
                          sess=sess)

In [ ]:
debiased_model.fit(dataset_orig_train)

epoch 0; iter: 892; batch classifier mean loss: 0.645735; batch adversarial mean loss: 0.691832


In [ ]:
# 将朴素模型应用于测试数据
dataset_debiasing_train = debiased_model.predict(dataset_orig_train)
dataset_debiasing_test = debiased_model.predict(dataset_orig_test)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 1. 获取特征重要性并筛选前9个 + sex
print("特征重要性分析：")
feature_importance_debiased = debiased_model.get_feature_importance(dataset_orig_train)

# 排序特征重要性
sorted_importance_debiased = sorted(feature_importance_debiased, key=lambda x: x[1], reverse=True)

# 提取前9个重要特征
top_9_features = sorted_importance_debiased[:9]
top_9_names = [x[0] for x in top_9_features]

# 添加sex特征（如果不在前9个中）
selected_features = top_9_names.copy()
if 'sex' not in selected_features:
    # 找到sex的重要性分数
    sex_importance = next((x for x in feature_importance_debiased if x[0] == 'sex'), None)
    if sex_importance:
        selected_features.append('sex')
        # 重新构建selected_importance列表
        selected_importance = top_9_features + [sex_importance]
    else:
        print("警告：未找到sex特征")
        selected_importance = top_9_features
else:
    selected_importance = top_9_features

print(f"选中的特征（{len(selected_features)}个）：", selected_features)

In [ ]:
# 2. 绘制筛选后的特征重要性柱状图
plt.figure(figsize=(12, 8))
feature_names_filtered = [x[0] for x in selected_importance]
importance_values_filtered = [x[1] for x in selected_importance]

plt.barh(feature_names_filtered, importance_values_filtered)
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('Feature Importance (Top9 + Sensitive in FairEduNet)')
plt.tight_layout()
plt.show()

In [ ]:
# 3. 绘制筛选后的专家网络热力图
def visualize_expert_importance_filtered(model, dataset, selected_features, title):
    expert_importance = model.analyze_expert_weights()
    feature_names = dataset.feature_names
    
    if not expert_importance or not any(expert_importance):
        print(f"警告：无法获取{title}的专家网络重要性")
        return
    
    # 找到选中特征在原始特征列表中的索引
    feature_indices = []
    for feature in selected_features:
        try:
            idx = list(feature_names).index(feature)
            feature_indices.append(idx)
        except ValueError:
            print(f"警告：特征 '{feature}' 不在数据集中")
            continue
    
    # 提取选中特征的专家权重
    expert_filtered = []
    for expert_id, expert_layers in enumerate(expert_importance):
        if expert_layers:
            # 只使用第一层权重，并筛选选中的特征
            first_layer_weights = expert_layers[0]
            filtered_weights = [first_layer_weights[i] for i in feature_indices]
            expert_filtered.append(filtered_weights)
        else:
            # 如果该专家没有权重，用零填充
            expert_filtered.append(np.zeros(len(feature_indices)))
    
    # 创建热图
    plt.figure(figsize=(12, 8))
    ax = plt.gca()
    im = ax.imshow(expert_filtered, aspect='auto', cmap='viridis')
    
    # 设置坐标轴
    filtered_feature_names = [feature_names[i] for i in feature_indices]
    ax.set_xticks(np.arange(len(filtered_feature_names)))
    ax.set_yticks(np.arange(len(expert_filtered)))
    ax.set_xticklabels(filtered_feature_names, rotation=45, ha='right')
    ax.set_yticklabels([f'Expert {i}' for i in range(len(expert_filtered))])
    
    # 在每个格子中添加数值
    for i in range(len(expert_filtered)):
        for j in range(len(filtered_feature_names)):
            text = ax.text(j, i, f'{expert_filtered[i][j]:.3f}',
                         ha="center", va="center", color="white", fontsize=8)
    
    # 添加颜色条
    plt.colorbar(im, ax=ax, label='Feature Importance')
    
    # 设置标题和标签
    plt.title(f'{title} - MoE Feature Importance Heatmap')
    plt.tight_layout()
    plt.show()


In [ ]:
# 调用可视化函数
visualize_expert_importance_filtered(debiased_model, dataset_orig_train, selected_features, "FairEduNet")

In [ ]:
# # 去偏见的模型数据集的度量指标
display(Markdown("#### Model - with debiasing - dataset metrics"))
metric_dataset_debiasing_train = BinaryLabelDatasetMetric(dataset_debiasing_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_debiasing_train.mean_difference())

metric_dataset_debiasing_test = BinaryLabelDatasetMetric(dataset_debiasing_test, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_debiasing_test.mean_difference())




display(Markdown("#### Model - with debiasing - classification metrics"))
classified_metric_debiasing_test = ClassificationMetric(dataset_orig_test, 
                                                 dataset_debiasing_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)
print("Test set: Classification accuracy = %f" % classified_metric_debiasing_test.accuracy())
TPR = classified_metric_debiasing_test.true_positive_rate()
TNR = classified_metric_debiasing_test.true_negative_rate()
bal_acc_debiasing_test = 0.5*(TPR+TNR)

print("Test set: Statistical parity difference = %f" % classified_metric_debiasing_test.statistical_parity_difference())
print("Test set: Equalized Odds difference = %f" % classified_metric_debiasing_test.equalized_odds_difference())
print("Test set: Equal opportunity difference = %f" % classified_metric_debiasing_test.equal_opportunity_difference())
print("Test set: Disparate impact = %f" % classified_metric_debiasing_test.disparate_impact())